# NB 06: Gradio Demo Prototype

**Goal:** Wire everything together into an interactive Gradio app with three tabs, ready for HF Spaces deployment.

| Tab | HF Library | What it does |
|-----|-----------|-------------|
| **Surface Explorer** | `gradio` + `plotly` | Browse real SPX surfaces by date, 3D interactive plot |
| **DDPM Generator** | `diffusers` | Generate new surfaces from noise, check arbitrage |
| **FinBERT Sentiment** | `transformers` | Score financial headlines, show sentiment breakdown |

This notebook builds and smoke-tests the app. To launch it interactively, run `demo.launch()` at the end.

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import numpy as np
import torch
import plotly.graph_objects as go
import gradio as gr
from transformers import pipeline as hf_pipeline

from hf_volsurf.data.loaders import VolSurfaceDataLoader
from hf_volsurf.utils.vol_math import STRIKE_GRID, TENOR_ORDER, tenor_to_years

OUTPUT_DIR = PROJECT_ROOT / "output"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print("=== NB 06: Gradio Demo Prototype ===\n")

PROJECT_ROOT: c:\source\repos\ale\huggin-face\hugging-face-learning
=== NB 06: Gradio Demo Prototype ===



## 1. Load data and models

Loading three components:
1. **Vol surface database** — 4,134 daily surfaces for the Explorer tab
2. **DDPM model** — trained UNet2D weights from NB 05 (651K params) for the Generator tab
3. **FinBERT** — pre-trained sentiment model from HuggingFace Hub for the Sentiment tab

In [2]:
loader = VolSurfaceDataLoader(PROJECT_ROOT / "data" / "db" / "hf_volsurf.db")
grids, dates = loader.get_all_surface_grids()
print(f"Loaded {len(dates)} surfaces")

# Load DDPM model
from diffusers import UNet2DModel, DDPMScheduler

ddpm_model = UNet2DModel(
    sample_size=(8, 16), in_channels=1, out_channels=1,
    layers_per_block=1, block_out_channels=(32, 64),
    down_block_types=("DownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "UpBlock2D"),
).to(DEVICE)

ddpm_weights = OUTPUT_DIR / "05_ddpm_model.pt"
if ddpm_weights.exists():
    ddpm_model.load_state_dict(torch.load(ddpm_weights, map_location=DEVICE, weights_only=True))
    ddpm_model.eval()
    print("DDPM model loaded")
else:
    print("WARNING: DDPM model not found, generation tab won't work")

grid_min, grid_max = grids.min(), grids.max()

noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",
)

# Load FinBERT
print("Loading FinBERT...")
sentiment = hf_pipeline("sentiment-analysis", model="ProsusAI/finbert", top_k=None)
print("FinBERT loaded")

Loaded 4134 surfaces


c:\source\repos\ale\.venv\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


DDPM model loaded
Loading FinBERT...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FinBERT loaded


## 2. Helper functions

- **`plot_surface_3d()`** — Creates interactive 3D plotly surface plots. Plotly is natively supported by Gradio's `gr.Plot` component.
- **`generate_surface()`** — Runs the full 1000-step DDPM reverse process on GPU, strips padding, denormalises to IV units.

In [3]:
strikes = np.array(STRIKE_GRID)
tenors_years = np.array([tenor_to_years(t) for t in TENOR_ORDER])


def plot_surface_3d(grid, title="Vol Surface"):
    """Create a 3D plotly surface plot."""
    X, Y = np.meshgrid(strikes, tenors_years)
    fig = go.Figure(data=[go.Surface(
        x=X, y=Y, z=grid,
        colorscale="Viridis",
        colorbar=dict(title="IV"),
    )])
    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title="Moneyness (K/S)",
            yaxis_title="Tenor (years)",
            zaxis_title="Implied Vol",
        ),
        width=600, height=500,
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig


def generate_surface():
    """Generate a new surface using DDPM."""
    with torch.no_grad():
        sample = torch.randn(1, 1, 8, 16).to(DEVICE)
        noise_scheduler.set_timesteps(1000)
        for t in noise_scheduler.timesteps:
            pred = ddpm_model(sample, t).sample
            sample = noise_scheduler.step(pred, t, sample).prev_sample
    gen = sample.cpu().numpy()[0, 0, :, :13]
    return gen * (grid_max - grid_min) + grid_min

## 3. Tab functions

Each tab is a Python function that takes user input and returns (plot, text_info):

- **Tab 1 (Explorer):** Queries the database for a date, extracts ATM vol, skew, and IV range, renders a 3D surface.
- **Tab 2 (Generator):** Generates N surfaces via DDPM, runs arbitrage checks on each, shows the first surface as 3D plot.
- **Tab 3 (Sentiment):** Runs FinBERT on a headline, returns a bar chart of positive/negative/neutral scores and the net sentiment.

In [4]:
def tab1_explore(date_str):
    """Surface Explorer tab."""
    grid = loader.get_surface_as_grid(date_str)
    if grid is None:
        return go.Figure(), f"No data for {date_str}"
    atm_idx = STRIKE_GRID.index(1.0)
    atm_1m = grid[0, atm_idx]
    atm_1y = grid[5, atm_idx]
    skew = grid[0, STRIKE_GRID.index(0.9)] - grid[0, STRIKE_GRID.index(1.1)]
    info = (
        f"Date: {date_str}\n"
        f"ATM 1m IV: {atm_1m:.4f} ({atm_1m:.2%})\n"
        f"ATM 1y IV: {atm_1y:.4f} ({atm_1y:.2%})\n"
        f"1m Skew (90-110%): {skew:.4f}\n"
        f"Min IV: {grid.min():.4f}, Max IV: {grid.max():.4f}"
    )
    return plot_surface_3d(grid, f"SPX Vol Surface — {date_str}"), info


def tab2_generate(n_surfaces):
    """DDPM Generator tab."""
    n = int(n_surfaces)
    surfaces = [generate_surface() for _ in range(n)]
    from hf_volsurf.utils.vol_math import check_calendar_arbitrage, check_butterfly_arbitrage
    cal_ok = sum(check_calendar_arbitrage(s, TENOR_ORDER) for s in surfaces)
    but_ok = sum(check_butterfly_arbitrage(s, STRIKE_GRID) for s in surfaces)

    fig = plot_surface_3d(surfaces[0], "DDPM Generated Surface (sample 1)")
    info = (
        f"Generated {n} surfaces\n"
        f"Calendar arb free: {cal_ok}/{n}\n"
        f"Butterfly arb free: {but_ok}/{n}\n"
        f"IV range: [{min(s.min() for s in surfaces):.4f}, {max(s.max() for s in surfaces):.4f}]"
    )
    return fig, info


def tab3_sentiment(headline):
    """FinBERT Sentiment tab."""
    results = sentiment([headline])[0]
    score_dict = {s["label"]: s["score"] for s in results}
    net = score_dict.get("positive", 0) - score_dict.get("negative", 0)

    # Bar chart of scores
    fig = go.Figure(data=[go.Bar(
        x=list(score_dict.keys()),
        y=list(score_dict.values()),
        marker_color=["green", "red", "gray"],
    )])
    fig.update_layout(
        title=f"FinBERT Sentiment (net: {net:+.3f})",
        yaxis_title="Score",
        width=500, height=350,
    )

    info = (
        f"Headline: {headline}\n"
        f"Positive: {score_dict.get('positive', 0):.4f}\n"
        f"Negative: {score_dict.get('negative', 0):.4f}\n"
        f"Neutral:  {score_dict.get('neutral', 0):.4f}\n"
        f"Net sentiment: {net:+.4f}"
    )
    return fig, info

## 4. Build Gradio app

Using `gr.Blocks()` for a multi-tab layout. Each tab has inputs (textbox, slider, button) and outputs (plotly plot, text info). The `click` handlers wire the tab functions to the UI components.

The app is built but not launched here — call `demo.launch()` to start the local server.

In [5]:
print("\nBuilding Gradio app...")

with gr.Blocks(title="HF VolSurf Explorer") as demo:
    gr.Markdown("# HF VolSurf: Volatility Surface Explorer")
    gr.Markdown("Explore SPX implied volatility surfaces using Hugging Face models.")

    with gr.Tab("Surface Explorer"):
        gr.Markdown("### Browse real SPX vol surfaces (2010-2026)")
        with gr.Row():
            date_input = gr.Textbox(
                value="2020-03-16", label="Date (YYYY-MM-DD)",
                info=f"Available: {dates[0]} to {dates[-1]}"
            )
            explore_btn = gr.Button("Load Surface")
        with gr.Row():
            surface_plot = gr.Plot(label="3D Vol Surface")
            surface_info = gr.Textbox(label="Surface Info", lines=6)
        explore_btn.click(tab1_explore, inputs=date_input, outputs=[surface_plot, surface_info])

    with gr.Tab("DDPM Generator"):
        gr.Markdown("### Generate new vol surfaces from noise using DDPM")
        with gr.Row():
            n_gen = gr.Slider(1, 20, value=5, step=1, label="Number of surfaces")
            gen_btn = gr.Button("Generate")
        with gr.Row():
            gen_plot = gr.Plot(label="Generated Surface")
            gen_info = gr.Textbox(label="Generation Info", lines=5)
        gen_btn.click(tab2_generate, inputs=n_gen, outputs=[gen_plot, gen_info])

    with gr.Tab("FinBERT Sentiment"):
        gr.Markdown("### Analyze financial headline sentiment with FinBERT")
        with gr.Row():
            headline_input = gr.Textbox(
                value="Federal Reserve raises rates by 75 basis points",
                label="Financial Headline",
            )
            sent_btn = gr.Button("Analyze")
        with gr.Row():
            sent_plot = gr.Plot(label="Sentiment Scores")
            sent_info = gr.Textbox(label="Analysis", lines=6)
        sent_btn.click(tab3_sentiment, inputs=headline_input, outputs=[sent_plot, sent_info])

print("Gradio app built successfully.")
print("\nTo launch interactively, run:")
print("  cd huggin-face/hugging-face-learning")
print("  poetry run python -c \"exec(open('notebooks/06_gradio_demo.py').read()); demo.launch()\"")
print("\nOr from app/app.py for deployment.\n")


Building Gradio app...
Gradio app built successfully.

To launch interactively, run:
  cd huggin-face/hugging-face-learning
  poetry run python -c "exec(open('notebooks/06_gradio_demo.py').read()); demo.launch()"

Or from app/app.py for deployment.



## 5. Smoke test

Call each tab function directly to verify everything works without launching the server.

**Results:**
- **Explorer:** COVID crash date (2020-03-16) → ATM 1m IV = **77.51%** — the highest in the dataset
- **Generator:** 3 surfaces generated successfully
- **Sentiment:** "Markets crash on recession fears" → net sentiment = **-0.92** — correctly identified as strongly negative

All three HF libraries (`transformers`, `diffusers`, `gradio`) working together in a single app.

In [6]:
print("--- Smoke testing all tabs ---")

fig1, info1 = tab1_explore("2020-03-16")
print(f"Tab 1 (Explorer): {info1.split(chr(10))[1]}")

fig2, info2 = tab2_generate(3)
print(f"Tab 2 (Generator): {info2.split(chr(10))[0]}")

fig3, info3 = tab3_sentiment("Markets crash on recession fears")
print(f"Tab 3 (Sentiment): {info3.split(chr(10))[-1]}")

print("\n=== NB 06 Complete ===")
print("Gradio app with 3 tabs: Surface Explorer, DDPM Generator, FinBERT Sentiment")
print("Ready for deployment to HF Spaces.")

--- Smoke testing all tabs ---
Tab 1 (Explorer): ATM 1m IV: 0.7751 (77.51%)
Tab 2 (Generator): Generated 3 surfaces
Tab 3 (Sentiment): Net sentiment: -0.9221

=== NB 06 Complete ===
Gradio app with 3 tabs: Surface Explorer, DDPM Generator, FinBERT Sentiment
Ready for deployment to HF Spaces.
